# Data Preprocessing and Feature Engineering

This notebook prepares the obesity-risk dataset for machine-learning model development.

## Objectives

- Load the raw dataset without modifying it
- Separate predictive features from the target
- Remove non-predictive identifiers
- Create stratified training, validation, and test datasets
- Define numerical, ordinal, and nominal feature groups
- Build reusable preprocessing pipelines
- Prevent data leakage

In [40]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [41]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System
Dataset path: c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System\data\raw\obesity.csv
Dataset exists: True


In [42]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

Dataset loaded successfully
Dataset shape: (20758, 18)


In [43]:
df.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [44]:
print("Columns:", df.columns.tolist())

Columns: ['id', 'Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']


In [45]:
identifier_column = "id"
target_column = "NObeyesdad"

print("Identifier column:", identifier_column)
print("Target column:", target_column)

Identifier column: id
Target column: NObeyesdad


In [46]:
X = df.drop(
    columns=[
        identifier_column,
        target_column,
    ]
)

y = df[target_column].copy()

In [47]:
print("Original dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Original dataset shape: (20758, 18)
Feature matrix shape: (20758, 16)
Target vector shape: (20758,)


In [48]:
print("Predictive features:")

for number, column in enumerate(X.columns, start=1):
    print(f"{number}. {column}")

Predictive features:
1. Gender
2. Age
3. Height
4. Weight
5. family_history_with_overweight
6. FAVC
7. FCVC
8. NCP
9. CAEC
10. SMOKE
11. CH2O
12. SCC
13. FAF
14. TUE
15. CALC
16. MTRANS


In [49]:
print(
    "Identifier present in X:",
    identifier_column in X.columns,
)

print(
    "Target present in X:",
    target_column in X.columns,
)

Identifier present in X: False
Target present in X: False


In [50]:
print("Number of target classes:", y.nunique())
print("Target classes:")

for class_name in sorted(y.unique()):
    print(class_name)

Number of target classes: 7
Target classes:
Insufficient_Weight
Normal_Weight
Obesity_Type_I
Obesity_Type_II
Obesity_Type_III
Overweight_Level_I
Overweight_Level_II


In [51]:
print("X row count:", len(X))
print("y row count:", len(y))
print("Indexes match:", X.index.equals(y.index))

X row count: 20758
y row count: 20758
Indexes match: True


In [52]:
numerical_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE",
]

In [53]:
ordinal_features = [
    "CAEC",
    "CALC",
]

In [54]:
nominal_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "SMOKE",
    "SCC",
    "MTRANS",
]

In [55]:
print("Numerical features:")
print(numerical_features)

print("\nOrdinal categorical features:")
print(ordinal_features)

print("\nNominal categorical features:")
print(nominal_features)

print("\nFeature counts")
print("Numerical:", len(numerical_features))
print("Ordinal:", len(ordinal_features))
print("Nominal:", len(nominal_features))

Numerical features:
['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

Ordinal categorical features:
['CAEC', 'CALC']

Nominal categorical features:
['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'MTRANS']

Feature counts
Numerical: 8
Ordinal: 2
Nominal: 6


In [56]:
grouped_features = (
    numerical_features
    + ordinal_features
    + nominal_features
)

missing_features = set(X.columns) - set(grouped_features)
unexpected_features = set(grouped_features) - set(X.columns)

print("Total grouped features:", len(grouped_features))
print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)

Total grouped features: 16
Missing features: set()
Unexpected features: set()


In [57]:
feature_group_sets = {
    "numerical": set(numerical_features),
    "ordinal": set(ordinal_features),
    "nominal": set(nominal_features),
}

numerical_ordinal_overlap = (
    feature_group_sets["numerical"]
    & feature_group_sets["ordinal"]
)

numerical_nominal_overlap = (
    feature_group_sets["numerical"]
    & feature_group_sets["nominal"]
)

ordinal_nominal_overlap = (
    feature_group_sets["ordinal"]
    & feature_group_sets["nominal"]
)

print(
    "Numerical and ordinal overlap:",
    numerical_ordinal_overlap,
)

print(
    "Numerical and nominal overlap:",
    numerical_nominal_overlap,
)

print(
    "Ordinal and nominal overlap:",
    ordinal_nominal_overlap,
)

Numerical and ordinal overlap: set()
Numerical and nominal overlap: set()
Ordinal and nominal overlap: set()


In [58]:
assert len(grouped_features) == len(set(grouped_features)), (
    "A feature appears in more than one group."
)

assert set(grouped_features) == set(X.columns), (
    "Feature groups do not match the predictive columns."
)

print("Feature group validation passed")

Feature group validation passed


In [60]:
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
TEMPORARY_SIZE = 0.30

print("Total split proportion:", TRAIN_SIZE + TEMPORARY_SIZE)

Total split proportion: 1.0


In [61]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    train_size=TRAIN_SIZE,
    test_size=TEMPORARY_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [62]:
print("Full feature set:", X.shape)
print("Full target set:", y.shape)

print("\nTraining features:", X_train.shape)
print("Training target:", y_train.shape)

print("\nTemporary features:", X_temp.shape)
print("Temporary target:", y_temp.shape)

Full feature set: (20758, 16)
Full target set: (20758,)

Training features: (14530, 16)
Training target: (14530,)

Temporary features: (6228, 16)
Temporary target: (6228,)


In [63]:
training_percentage = len(X_train) / len(X) * 100
temporary_percentage = len(X_temp) / len(X) * 100

print(f"Training percentage: {training_percentage:.2f}%")
print(f"Temporary percentage: {temporary_percentage:.2f}%")

Training percentage: 70.00%
Temporary percentage: 30.00%


In [64]:
print(
    "Training indexes match:",
    X_train.index.equals(y_train.index),
)

print(
    "Temporary indexes match:",
    X_temp.index.equals(y_temp.index),
)

Training indexes match: True
Temporary indexes match: True


In [65]:
overlapping_indexes = (
    set(X_train.index)
    & set(X_temp.index)
)

print(
    "Number of overlapping records:",
    len(overlapping_indexes),
)

Number of overlapping records: 0


In [66]:
assert len(X_train) == len(y_train), (
    "Training features and targets have different lengths"
)

assert len(X_temp) == len(y_temp), (
    "Temporary features and targets have different lengths"
)

assert len(X_train) + len(X_temp) == len(X), (
    "The split does not contain every original record"
)

assert not overlapping_indexes, (
    "Training and temporary sets contain overlapping records"
)

print("Initial split validation passed")

Initial split validation passed
